# End-to-end demo

Load the saved GMM + logistic regression and run the full pipeline on the validation set: probability, band label, and a lesion-overlay gallery.

In [ ]:
from _setup import DATA_DIR, ARTIFACTS_DIR, ensure_dataset
import numpy as np
import matplotlib.pyplot as plt
from phytolabs import io, segmentation, pipeline, viz
from phytolabs.logreg import LogisticRegressionSGD

data_dir = ensure_dataset()
leaf_gmm = segmentation.LeafGMM.load(ARTIFACTS_DIR / 'gmm.joblib')
model = LogisticRegressionSGD.load(ARTIFACTS_DIR / 'logreg.joblib')

## Predict on a few validation images

In [ ]:
samples = []
for cls in ('rust', 'healthy'):
    for p in list(io.iter_image_paths(data_dir / 'val' / cls))[:4]:
        result, seg, bgr = pipeline.predict_image(p, leaf_gmm, model)
        caption = f"{cls}: P={result['probability']:.2f} ({result['label']})"
        samples.append((bgr, seg['rust'], caption))
        print(caption, '|', p.name)

## Lesion-overlay gallery (the product UX)

In [ ]:
viz.overlay_gallery(samples, ncols=4)
plt.show()

## Scope reminder

This is **image-level** brown-rust classification. The overlays are a qualitative bonus from the unsupervised GMM; we do **not** claim per-region accuracy because the datasets only provide image-level labels.